# EXP002 Two-Stem Objective Evaluation

## Purpose
This notebook evaluates the existing authoritative four-stem HT-Demucs predictions under a derived two-stem formulation:
- vocals
- accompaniment = bass + drums + other

The accompaniment stem is constructed in memory from the three corresponding source predictions and evaluated against the equally constructed MUSDB18 ground-truth accompaniment reference. No authoritative four-stem prediction files are modified or overwritten.

The objective metrics are:
- SDR
- SIR
- SAR
- SI-SDR

This notebook represents a separate analytical branch from the authoritative four-stem evaluation.


In [1]:
# CELL 2 - DEPENDENCY INSTALLATION
!pip install -q museval torchmetrics torchaudio numpy pandas matplotlib torch soundfile torchcodec


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# CELL 3 - RUNTIME AND DEPENDENCY VALIDATION
import json
import math
import warnings
from pathlib import Path
import traceback
import importlib.metadata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torchaudio
import torch
import museval
import soundfile as sf
from torchmetrics.functional.audio.sdr import scale_invariant_signal_distortion_ratio

print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"TorchAudio: {torchaudio.__version__}")
print(f"Museval: {importlib.metadata.version('museval')}")
print(f"TorchMetrics: {importlib.metadata.version('torchmetrics')}")
print(f"SoundFile: {sf.__version__}")
print("All required metric engines are available.")
import io

SyntaxError: unexpected character after line continuation character (2325854688.py, line 26)

In [ ]:
# CELL 4 - PATH CONFIGURATION
EXP_NAME = "EXP002"
PROJECT_ROOT = Path(r"D:\IKS_Research")

# Authoritative Prediction Location
PREDICTIONS_ROOT = PROJECT_ROOT / "Evaluation" / "Objective" / EXP_NAME / "predictions"

# Authoritative Reference Location
MUSDB18_ROOT = PROJECT_ROOT / "MUSDB18" / "test"

# Two Source Outputs Location
TWO_SOURCE_ROOT = PROJECT_ROOT / "Evaluation" / "Two_Source" / EXP_NAME
METRICS_DIR = TWO_SOURCE_ROOT / "metrics"
REPORTS_DIR = TWO_SOURCE_ROOT / "reports"
PLOTS_DIR = TWO_SOURCE_ROOT / "plots"

for d in [METRICS_DIR, REPORTS_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Predictions Root: {PREDICTIONS_ROOT}")
print(f"MUSDB18 Root: {MUSDB18_ROOT}")
print(f"Two Source Root: {TWO_SOURCE_ROOT}")

assert PREDICTIONS_ROOT.exists(), f"Prediction root does not exist: {PREDICTIONS_ROOT}"
assert MUSDB18_ROOT.exists(), f"MUSDB18 reference root does not exist: {MUSDB18_ROOT}"

In [ ]:
# CELL 5 - AUTHORITATIVE MUSDB18 DISCOVERY
TRACK_REGISTRY = []
required_sources = ['vocals.wav', 'bass.wav', 'drums.wav', 'other.wav']

for track_dir in MUSDB18_ROOT.iterdir():
    if not track_dir.is_dir():
        continue
    missing = []
    sources_paths = {}
    for src in required_sources:
        src_path = track_dir / src
        if not src_path.exists():
            missing.append(src)
        else:
            sources_paths[src.split('.')[0]] = src_path
    
    if not missing:
        TRACK_REGISTRY.append({
            'track_name': track_dir.name,
            'ref_dir': track_dir,
            'ref_sources': sources_paths
        })
    else:
        print(f"Skipping MUSDB18 track {track_dir.name}: missing {missing}")

if not TRACK_REGISTRY:
    print("WARNING: Empty discovery. No valid MUSDB18 tracks found.")
else:
    print(f"Discovered {len(TRACK_REGISTRY)} valid MUSDB18 reference tracks containing all required sources.")

In [ ]:
# CELL 6 - AUTHORITATIVE PREDICTION DISCOVERY
COMPLETE_PREDICTIONS = []
missing_tracks = 0
incomplete_predictions = 0

for track_meta in TRACK_REGISTRY:
    track_name = track_meta['track_name']
    pred_dir = PREDICTIONS_ROOT / track_name
    
    if not pred_dir.exists():
        missing_tracks += 1
        print(f"Missing authoritative prediction track: {track_name}")
        continue
        
    missing_sources = []
    pred_paths = {}
    for src in required_sources:
        src_path = pred_dir / src
        if not src_path.exists():
            missing_sources.append(src)
        else:
            pred_paths[src.split('.')[0]] = src_path
            
    if missing_sources:
        incomplete_predictions += 1
        print(f"Incomplete prediction set {track_name}: missing {missing_sources}")
    else:
        track_meta['pred_dir'] = pred_dir
        track_meta['pred_sources'] = pred_paths
        COMPLETE_PREDICTIONS.append(track_meta)

print(f"Complete prediction sets: {len(COMPLETE_PREDICTIONS)}")
print(f"Incomplete prediction sets: {incomplete_predictions}")
print(f"Missing tracks relative to reference: {missing_tracks}")

In [ ]:
# CELL 7 - AUDIO COMPATIBILITY VALIDATION
VALIDATED_TRACKS = []

for track in COMPLETE_PREDICTIONS:
    is_valid = True
    ref_info_dict = {}
    pred_info_dict = {}
    
    # Check Ref Sources
    for src_name, path in track['ref_sources'].items():
        info = sf.info(path)
        ref_info_dict[src_name] = {'samplerate': info.samplerate, 'channels': info.channels, 'frames': info.frames}
    
    # Check Pred Sources
    for src_name, path in track['pred_sources'].items():
        info = sf.info(path)
        pred_info_dict[src_name] = {'samplerate': info.samplerate, 'channels': info.channels, 'frames': info.frames}
        
    # Validate against reference vocals
    base_sr = ref_info_dict['vocals']['samplerate']
    base_ch = ref_info_dict['vocals']['channels']
    
    for src_name in ['vocals', 'bass', 'drums', 'other']:
        if pred_info_dict[src_name]['samplerate'] != base_sr or pred_info_dict[src_name]['channels'] != base_ch:
            print(f"Track {track['track_name']} has channel/samplerate mismatch in {src_name}.")
            is_valid = False
            
    if is_valid:
        VALIDATED_TRACKS.append(track)

print(f"Validated tracks with compatible audio dimensions: {len(VALIDATED_TRACKS)}")
if len(VALIDATED_TRACKS) == 0:
    raise RuntimeError("Zero valid tracks found. Stopping execution to prevent downstream failure.")

In [ ]:
# CELL 8 - TWO-STEM CONSTRUCTION VALIDATION
# We perform a dry-run construction on the first track to validate shapes
import io
import soundfile as sf

if len(VALIDATED_TRACKS) == 0:
    print("No validated tracks available for dry-run.")
else:
    test_track = VALIDATED_TRACKS[0]
    
    def load_np(path):
        with open(path, 'rb') as f:
            w, _ = sf.read(io.BytesIO(f.read()), dtype='float32', always_2d=True)
        return w

    w_b = load_np(test_track['pred_sources']['bass'])
    w_d = load_np(test_track['pred_sources']['drums'])
    w_o = load_np(test_track['pred_sources']['other'])
    w_v = load_np(test_track['pred_sources']['vocals'])
    
    min_acc = min(w_b.shape[0], w_d.shape[0], w_o.shape[0])
    w_acc = w_b[:min_acc] + w_d[:min_acc] + w_o[:min_acc]
    
    print(f"Validation Track: {test_track['track_name']}")
    print(f"Vocals Shape: {w_v.shape}")
    print(f"Accompaniment Shape: {w_acc.shape}")


In [ ]:
# CELL 9 - METRIC ENGINE CONFIGURATION
METRIC_CONFIG = {
    'museval_version': importlib.metadata.version('museval'),
    'torchmetrics_version': importlib.metadata.version('torchmetrics'),
    'metrics': ['SDR', 'SIR', 'SAR', 'SI-SDR'],
    'window': 'framewise',  # BSS Eval standard
    'permutation': False    # Disabled in standard 4-stem and 2-stem evaluations
}
print(json.dumps(METRIC_CONFIG, indent=2))

In [ ]:
# CELL 10 - PERSISTENT REGISTRY / RESULT STATE
EXP002_TWO_SOURCE_CONFIG = METRIC_CONFIG
EXP002_TWO_SOURCE_TRACKS = VALIDATED_TRACKS
EXP002_TWO_SOURCE_RESULTS_PATH = METRICS_DIR / "two_source_results.jsonl"

In [ ]:
# CELL 11 - LOAD COMPLETED TRACKS FOR RESUMABILITY
import json
from pathlib import Path

# Create metrics directory if it doesn't exist
Path(EXP002_TWO_SOURCE_RESULTS_PATH).parent.mkdir(parents=True, exist_ok=True)

# Maintain a set of (track_name, source_name) tuples
COMPLETED_SOURCES = set()

if Path(EXP002_TWO_SOURCE_RESULTS_PATH).exists():
    with open(EXP002_TWO_SOURCE_RESULTS_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                record = json.loads(line)
                t_name = record.get('track')
                s_name = record.get('source')
                if t_name and s_name:
                    COMPLETED_SOURCES.add((t_name, s_name))
            except json.JSONDecodeError:
                pass

print(f"Resuming evaluation: {len(COMPLETED_SOURCES)} source records already completed.")


In [ ]:
# CELL 12 - TWO-STEM OBJECTIVE EVALUATION
import warnings
import io
import traceback

def load_np(path):
    w, _ = sf.read(str(path), dtype='float32', always_2d=True)
    return w

# Open in append mode for resumability
with open(EXP002_TWO_SOURCE_RESULTS_PATH, 'a', encoding='utf-8') as f:
    for track in EXP002_TWO_SOURCE_TRACKS:
        t_name = track['track_name']
        
        # Check if BOTH sources are already completed
        if (t_name, 'vocals') in COMPLETED_SOURCES and (t_name, 'accompaniment') in COMPLETED_SOURCES:
            continue
            
        try:
            # ----------------------------------------------------------------------
            # 1. LOAD ALL 8 FILES
            # ----------------------------------------------------------------------
            r_voc_raw = load_np(track['ref_sources']['vocals'])
            r_bass_raw = load_np(track['ref_sources']['bass'])
            r_drums_raw = load_np(track['ref_sources']['drums'])
            r_other_raw = load_np(track['ref_sources']['other'])
            
            p_voc_raw = load_np(track['pred_sources']['vocals'])
            p_bass_raw = load_np(track['pred_sources']['bass'])
            p_drums_raw = load_np(track['pred_sources']['drums'])
            p_other_raw = load_np(track['pred_sources']['other'])
            
            # ----------------------------------------------------------------------
            # 2. INTRA-SOURCE LENGTH ALIGNMENT
            # ----------------------------------------------------------------------
            # Accompaniment Reference
            min_r_acc = min(r_bass_raw.shape[0], r_drums_raw.shape[0], r_other_raw.shape[0])
            r_acc_raw = r_bass_raw[:min_r_acc] + r_drums_raw[:min_r_acc] + r_other_raw[:min_r_acc]
            
            # Accompaniment Prediction
            min_p_acc = min(p_bass_raw.shape[0], p_drums_raw.shape[0], p_other_raw.shape[0])
            p_acc_raw = p_bass_raw[:min_p_acc] + p_drums_raw[:min_p_acc] + p_other_raw[:min_p_acc]
            
            # ----------------------------------------------------------------------
            # 3. INTER-SOURCE LENGTH ALIGNMENT
            # ----------------------------------------------------------------------
            # Now align Vocals and Accompaniment across Reference and Prediction
            min_len = min(
                r_voc_raw.shape[0], r_acc_raw.shape[0],
                p_voc_raw.shape[0], p_acc_raw.shape[0]
            )
            
            r_voc = r_voc_raw[:min_len]
            r_acc = r_acc_raw[:min_len]
            
            p_voc = p_voc_raw[:min_len]
            p_acc = p_acc_raw[:min_len]
            
            # Basic Safety Validations
            if r_voc.shape != p_voc.shape or r_acc.shape != p_acc.shape:
                raise RuntimeError("Shape mismatch after alignment.")
                
            if r_voc.ndim != 2 or r_acc.ndim != 2 or p_voc.ndim != 2 or p_acc.ndim != 2:
                raise RuntimeError("Arrays are not 2-dimensional.")
                
            # Assert NaN/Inf
            if np.isnan(r_voc).any() or np.isnan(r_acc).any() or np.isnan(p_voc).any() or np.isnan(p_acc).any():
                raise RuntimeError("NaN found in audio arrays.")
            if np.isinf(r_voc).any() or np.isinf(r_acc).any() or np.isinf(p_voc).any() or np.isinf(p_acc).any():
                raise RuntimeError("Inf found in audio arrays.")
                
            # Assert non-silent references
            if np.all(r_voc == 0) or np.all(r_acc == 0):
                raise RuntimeError("Silent reference source detected (all zeros).")

            # ----------------------------------------------------------------------
            # 4. STACK SOURCES: (sources, frames, channels)
            # Index 0: vocals, Index 1: accompaniment
            # ----------------------------------------------------------------------
            references = np.stack([r_voc, r_acc], axis=0).astype(np.float32, copy=False)
            estimates = np.stack([p_voc, p_acc], axis=0).astype(np.float32, copy=False)

            # ----------------------------------------------------------------------
            # 5. BSSEVAL V4 (window=np.inf)
            # ----------------------------------------------------------------------
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                sdr, isr, sir, sar, _ = museval.metrics.bss_eval(
                    references,
                    estimates,
                    window=np.inf,
                    hop=np.inf,
                    compute_permutation=False,
                    filters_len=512,
                    framewise_filters=False,
                    bsseval_sources_version=False,
                )
            
            # In time-invariant evaluation (window=np.inf), the returned array shapes
            # are (sources, 1). E.g. sdr[0, 0] is SDR for Vocals.
            
            # ----------------------------------------------------------------------
            # 6. SI-SDR
            # ----------------------------------------------------------------------
            si_sdr_values = []
            metric = scale_invariant_signal_distortion_ratio
            for i in range(2):
                # Shape requirement for SI-SDR: (batch, channels, samples)
                r_tensor = torch.from_numpy(np.asarray(references[i].T, dtype=np.float32)).unsqueeze(0)
                p_tensor = torch.from_numpy(np.asarray(estimates[i].T, dtype=np.float32)).unsqueeze(0)
                with torch.no_grad():
                    val = metric(p_tensor, r_tensor).detach().cpu().numpy()
                si_sdr_values.append(float(val.reshape(-1)[0]))
                
            # ----------------------------------------------------------------------
            # 7. WRITE RESULTS
            # ----------------------------------------------------------------------
            source_names = ['vocals', 'accompaniment']
            
            for i, source_name in enumerate(source_names):
                if (t_name, source_name) not in COMPLETED_SOURCES:
                    record = {
                        'experiment': EXP_NAME,
                        'track': t_name,
                        'source': source_name,
                        'metrics': {
                            'SDR': float(sdr[i, 0]),
                            'SIR': float(sir[i, 0]),
                            'SAR': float(sar[i, 0]),
                            'SI-SDR': float(si_sdr_values[i])
                        }
                    }
                    f.write(json.dumps(record) + "\n")
                    f.flush()
                    COMPLETED_SOURCES.add((t_name, source_name))
                    print(f"[{source_name.upper()}] {t_name} evaluated.")
                    
        except Exception as e:
            print(f"Failed source metric calculation for {t_name}: {str(e)}")
            traceback.print_exc()

print("Evaluation phase completed.")


In [ ]:
# CELL 13 - METRIC INTEGRITY AUDIT
results = []
if EXP002_TWO_SOURCE_RESULTS_PATH.exists():
    with open(EXP002_TWO_SOURCE_RESULTS_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                results.append(json.loads(line.strip()))
            except:
                pass

df = pd.json_normalize(results)

if not df.empty:
    df = df.rename(columns={
        'metrics.SDR': 'SDR',
        'metrics.SIR': 'SIR',
        'metrics.SAR': 'SAR',
        'metrics.SI-SDR': 'SI-SDR'
    })

audit_passed = True
missing_metric_count = 0

if not df.empty:
    # Check expected schema
    expected_cols = ['experiment', 'track', 'source', 'SDR', 'SIR', 'SAR', 'SI-SDR']
    for c in expected_cols:
        if c not in df.columns:
            print(f"AUDIT FAIL: Missing column {c}")
            audit_passed = False
            
    # Check valid sources
    invalid_sources = df[~df['source'].isin(['vocals', 'accompaniment'])]
    if len(invalid_sources) > 0:
        print(f"AUDIT FAIL: Invalid sources found: {invalid_sources['source'].unique()}")
        audit_passed = False

    # Check duplicates
    dups = df.duplicated(subset=['track', 'source'])
    if dups.any():
        print(f"AUDIT FAIL: Duplicate (track, source) records found: {dups.sum()}")
        audit_passed = False
        
    # Check NaN/Inf values explicitly reporting them
    missing_metric_count = df[['SDR', 'SIR', 'SAR', 'SI-SDR']].isna().sum().sum()
    if missing_metric_count > 0:
        print(f"AUDIT WARNING: Found {missing_metric_count} missing or non-finite metric values.")
        # Note: We do not fail the audit just because a mathematically undefined case occurred
else:
    print("AUDIT FAIL: DataFrame is empty.")
    audit_passed = False

expected_count = len(EXP002_TWO_SOURCE_TRACKS) * 2
actual_count = len(df)
if actual_count != expected_count:
    print(f"AUDIT WARNING: Expected {expected_count} records, got {actual_count}.")
    if actual_count < expected_count:
        audit_passed = False # Fail if incomplete

if not df.empty:
    df.to_csv(REPORTS_DIR / 'metric_integrity_audit.csv', index=False)
print(f"Metric Integrity Audit: {'PASSED' if audit_passed else 'FAILED'}")

In [ ]:
# CELL 14 - DESCRIPTIVE STATISTICS
if not df.empty:
    stats_df = df.groupby('source')[['SDR', 'SIR', 'SAR', 'SI-SDR']].agg(['count', 'mean', 'median', 'std', 'min', 'max'])
    stats_df.to_csv(REPORTS_DIR / 'descriptive_statistics.csv')
    display(stats_df)
else:
    print("DataFrame empty, skipping statistics.")

In [ ]:
# CELL 15 - SOURCE-LEVEL ANALYSIS
if not df.empty:
    source_means = df.groupby('source')[['SDR', 'SIR', 'SAR', 'SI-SDR']].mean()
    source_means.to_csv(REPORTS_DIR / 'source_level_analysis.csv')
    display(source_means)

In [ ]:
# CELL 16 - DISTRIBUTION ANALYSIS
if not df.empty:
    # Pandas quantile automatically ignores NaNs
    q1 = df[['SDR', 'SIR', 'SAR', 'SI-SDR']].quantile(0.25)
    q3 = df[['SDR', 'SIR', 'SAR', 'SI-SDR']].quantile(0.75)
    iqr = q3 - q1
    
    dist_report = pd.DataFrame({
        'Q1': q1,
        'Q3': q3,
        'IQR': iqr,
        'Lower_Fence': q1 - 1.5 * iqr,
        'Upper_Fence': q3 + 1.5 * iqr
    })
    dist_report.to_csv(REPORTS_DIR / 'distribution_outlier_analysis.csv')
    display(dist_report)

In [ ]:
# CELL 17 - REQUIRED TWO-STEM PLOTS
if not df.empty:
    try:
        # 1. Source Comparison Plot
        fig, ax = plt.subplots(figsize=(10, 6))
        source_means.T.plot(kind='bar', ax=ax)
        ax.set_title(f"{EXP_NAME} - Two-Stem Source Comparison")
        ax.set_ylabel("dB")
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / 'source_comparison.png', dpi=300)
        plt.close()
        
        # 2. Metric Distribution (Boxplots)
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        axes = axes.flatten()
        for idx, metric in enumerate(['SDR', 'SIR', 'SAR', 'SI-SDR']):
            df.dropna(subset=[metric]).boxplot(column=metric, by='source', ax=axes[idx])
            axes[idx].set_title(metric)
            axes[idx].set_xlabel("")
        plt.suptitle(f"{EXP_NAME} - Metric Distributions")
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / 'metric_distributions.png', dpi=300)
        plt.close()
        
        print("Plots saved to plots directory.")
    except Exception as e:
        print(f"Plotting failed: {str(e)}")

In [ ]:
# CELL 18 - FINAL QC
qc_passed = True
if not audit_passed:
    print("QC FAIL: Metric Audit Failed.")
    qc_passed = False

if len(list(PLOTS_DIR.glob('*.png'))) < 2:
    print("QC FAIL: Expected plots are missing.")
    qc_passed = False

if len(list(REPORTS_DIR.glob('*.csv'))) < 4:
    print("QC FAIL: Expected CSV reports are missing.")
    qc_passed = False

print(f"Final QC: {'PASSED' if qc_passed else 'FAILED'}")

In [ ]:
# CELL 19 - COMPLETION MANIFEST
# Calculate fully evaluated tracks (those that have BOTH vocals and accompaniment records)
fully_evaluated = 0
if not df.empty:
    track_counts = df.groupby('track')['source'].nunique()
    fully_evaluated = (track_counts == 2).sum()

manifest = {
    "experiment": EXP_NAME,
    "evaluation_type": "two-stem objective",
    "input_prediction_root": str(PREDICTIONS_ROOT),
    "input_reference_root": str(MUSDB18_ROOT),
    "output_root": str(TWO_SOURCE_ROOT),
    "validated_tracks": len(VALIDATED_TRACKS),
    "source_records_generated": len(df) if not df.empty else 0,
    "evaluated_tracks": int(fully_evaluated),
    "qc_summary": "PASSED" if qc_passed else "FAILED",
    "completion_status": "COMPLETE" if qc_passed else "INCOMPLETE"
}

with open(REPORTS_DIR / 'two_source_completion_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

if qc_passed:
    print("COMPLETE")
else:
    print("FAILED QC")